# Cursor IDE — Agent Mode Test

Verify the full coding assistant experience in **Cursor Agent mode** with MaaS-hosted models and MCP tools.

This notebook will:
1. Dynamically discover your MaaS endpoint and MCP server URLs
2. Verify connectivity to all services
3. Generate the configuration for Cursor IDE
4. Guide you through Agent mode testing with expected screenshots

**Prerequisites:** Phases 0–3 completed (model deployed, MaaS gateway running, MCP servers registered)

## Step 1: Discover MaaS Endpoint

In [ ]:
import subprocess
import json
import urllib.request
import ssl

result = subprocess.run(
    ["kubectl", "get", "ingresses.config.openshift.io", "cluster",
     "-o", "jsonpath={.spec.domain}"],
    capture_output=True, text=True
)
CLUSTER_DOMAIN = result.stdout.strip()
MAAS_HOST = f"https://maas-api.apps.{CLUSTER_DOMAIN}"

token_result = subprocess.run(["oc", "whoami", "-t"], capture_output=True, text=True)
OC_TOKEN = token_result.stdout.strip()

ctx = ssl.create_default_context()
ctx.check_hostname = False
ctx.verify_mode = ssl.CERT_NONE

print(f"Cluster Domain: {CLUSTER_DOMAIN}")
print(f"MaaS Endpoint:  {MAAS_HOST}")
print(f"Models URL:     {MAAS_HOST}/v1")

## Step 2: Discover Available Models

In [ ]:
req = urllib.request.Request(
    f"{MAAS_HOST}/v1/models",
    headers={"Authorization": f"Bearer {OC_TOKEN}", "Content-Type": "application/json"}
)
try:
    with urllib.request.urlopen(req, context=ctx) as resp:
        models_data = json.loads(resp.read())
    print("Available Models:")
    for m in models_data.get("data", []):
        print(f"  - {m['id']}")
    MODEL_NAME = models_data["data"][0]["id"]
    print(f"\nDefault model: {MODEL_NAME}")
except Exception as e:
    print(f"Error: {e}")
    MODEL_NAME = "MODEL_NOT_FOUND"

## Step 3: Discover MCP Server URLs

In [ ]:
routes_result = subprocess.run(
    ["kubectl", "get", "httproute", "-n", "mcp-servers",
     "-l", "maas.opendatahub.io/managed=true",
     "-o", "jsonpath={range .items[*]}{.metadata.name}\n{end}"],
    capture_output=True, text=True
)

mcp_servers = {}
for line in routes_result.stdout.strip().split("\n"):
    if line:
        name = line.replace("mcp-route-", "")
        mcp_servers[name] = f"{MAAS_HOST}/mcp/{name}/sse"

print("MCP Servers via MaaS Gateway:")
print("=" * 60)
for name, url in mcp_servers.items():
    print(f"  {name:25s} → {url}")
print(f"\nTotal: {len(mcp_servers)} MCP servers registered")

## Step 4: Verify Connectivity

In [ ]:
print("Connectivity Check:")
print("=" * 60)

# Test model endpoint
try:
    test_req = urllib.request.Request(
        f"{MAAS_HOST}/v1/models",
        headers={"Authorization": f"Bearer {OC_TOKEN}"}
    )
    with urllib.request.urlopen(test_req, context=ctx) as resp:
        status = resp.status
    print(f"  [OK] Model endpoint ({MAAS_HOST}/v1) — HTTP {status}")
except Exception as e:
    print(f"  [FAIL] Model endpoint — {e}")

# Test each MCP server
for name, url in mcp_servers.items():
    try:
        mcp_req = urllib.request.Request(
            url, headers={"Authorization": f"Bearer {OC_TOKEN}"}
        )
        with urllib.request.urlopen(mcp_req, context=ctx, timeout=5) as resp:
            print(f"  [OK] MCP: {name} — HTTP {resp.status}")
    except Exception as e:
        err_msg = str(e)[:50]
        if "200" in err_msg or "SSE" in err_msg:
            print(f"  [OK] MCP: {name} — SSE stream active")
        else:
            print(f"  [WARN] MCP: {name} — {err_msg}")

print("\nAll green? Proceed to configure Cursor IDE below.")

## Step 5: Generate Cursor Configuration

Copy the output below into your Cursor settings.

In [ ]:
API_KEY = "sk-oai-YOUR-KEY"  # Replace with your actual MaaS API key

print("═" * 60)
print("CURSOR MODEL SETTINGS")
print("═" * 60)
print(f"  Base URL : {MAAS_HOST}/v1")
print(f"  API Key  : {API_KEY}")
print(f"  Model    : {MODEL_NAME}")
print()
print("═" * 60)
print("CURSOR MCP SETTINGS (.cursor/mcp.json)")
print("═" * 60)

cursor_mcp_config = {
    "mcpServers": {
        name: {
            "url": url,
            "headers": {"Authorization": f"Bearer {API_KEY}"}
        }
        for name, url in mcp_servers.items()
    }
}
print(json.dumps(cursor_mcp_config, indent=2))

## Step 6: Configure Cursor IDE

### 6a. Model Settings

Open **Cursor Settings → Models → OpenAI Compatible** and enter the values above.

![Cursor Model Settings](screenshots/01-cursor-model-settings.png)

### 6b. MCP Server Settings

Copy the JSON from Step 5 into `.cursor/mcp.json` in your project root.

![Cursor MCP Settings](screenshots/02-cursor-mcp-settings.png)

## Step 7: Test Agent Mode

### 7a. Open Agent Mode

1. Open Cursor Chat panel (`Cmd+L` / `Ctrl+L`)
2. Switch to **Agent** mode (dropdown at top of chat)
3. Verify the model shows your MaaS-hosted model

![Agent Mode Selection](screenshots/03-agent-mode-selection.png)

### 7b. Test Code Generation

Prompt:
```
Create a Python FastAPI endpoint that accepts a JSON payload with "text" field
and returns the word count.
```

**Expected:** Complete FastAPI app with proper imports, endpoint, and response model.

![Code Generation](screenshots/04-code-generation.png)

### 7c. Test MCP Tool Calling

Prompt:
```
Use the GitHub MCP server to list the latest 5 issues from the
hyogrin/rhoai-code-assistant-lab repository.
```

**Expected:** Agent invokes the GitHub MCP tool and returns structured issue data.

![MCP Tool Call](screenshots/05-mcp-tool-call.png)

### 7d. Test Multi-Step Agent Workflow

Prompt:
```
Look up the latest Context7 documentation for FastAPI, then create a REST API
with health check, CRUD endpoints for a "Task" model, and error handling.
```

**Expected:** Agent calls Context7 MCP → plans with sequential-thinking → generates code.

![Multi-Step Workflow](screenshots/06-multi-step-workflow.png)

## Step 8: Verification Summary

In [ ]:
print("Cursor Agent Mode — Verification Checklist")
print("=" * 60)
print(f"  Model Endpoint : {MAAS_HOST}/v1")
print(f"  Model Name     : {MODEL_NAME}")
print(f"  MCP Servers    : {len(mcp_servers)} registered")
print()
print("  [ ] Model settings configured in Cursor")
print("  [ ] .cursor/mcp.json created with MCP servers")
print("  [ ] Agent mode selected in chat panel")
print("  [ ] Code generation works")
print("  [ ] MCP tool calling works")
print("  [ ] Multi-step workflow works")
print()
print("Add screenshots to 6_ide_integration_test/screenshots/")

## Troubleshooting

| Issue | Cause | Fix |
|-------|-------|-----|
| "Model not found" | Incorrect model name | Check output of Step 2 |
| SSL error | Self-signed certificate | Add cluster CA or disable verification |
| MCP timeout | Network/firewall | Verify Step 4 shows [OK] |
| Empty response | Token limit or rate limit | Check MaaS rate limit settings |
| Tool not available | MCP not registered | Re-run `2_ai_gateway/2_enable_maas.ipynb` |